# Sales Data Exploratory Analysis
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  |  **Runs in:** Google Colab (no installs needed)

**Goal:** Explore sales data to find patterns and trends (time, category, region, discounts) and communicate them with charts that support data-driven decisions.

**How to use:** Run the cells from top to bottom (click a cell, press Shift + Enter). Read the markdown above each code cell to understand what it does.

## Step 1: Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Folders for the dataset and saved charts
os.makedirs("data", exist_ok=True)
os.makedirs("charts", exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"

print("Libraries ready. pandas", pd.__version__)

## Step 2: Get the dataset
**Option A (default): generate the dataset.** This creates `data/sales.csv` (about 3,000 orders from 2023 to 2025, with a few missing values and duplicate rows added on purpose so you can practise cleaning).

**Option B: use your own file.** In Colab click the folder icon on the left, then upload `sales.csv` into the `data` folder. Then skip the generator cell (or it will not overwrite your file, because it only runs if the file is missing).

In [ ]:
DATA_PATH = "data/sales.csv"

if not os.path.exists(DATA_PATH):
    np.random.seed(42)
    n = 3000

    # random order dates between 2023 and 2025, with a mild year-end boost
    all_days = pd.date_range("2023-01-01", "2025-12-31")
    weights = np.where(all_days.month.isin([10, 11, 12]), 1.6, 1.0)
    weights = weights / weights.sum()
    dates = np.random.choice(all_days, n, p=weights)

    categories = {
        "Electronics": ["Phones", "Laptops", "Accessories"],
        "Furniture": ["Chairs", "Tables", "Bookcases"],
        "Office Supplies": ["Paper", "Binders", "Pens"],
    }
    cat_list = np.random.choice(list(categories.keys()), n)
    sub_list = [np.random.choice(categories[c]) for c in cat_list]

    raw = pd.DataFrame({
        "Order ID": [f"ORD-{i:05d}" for i in range(n)],
        "Order Date": pd.to_datetime(dates),
        "Region": np.random.choice(["North", "South", "East", "West"], n),
        "Category": cat_list,
        "Sub-Category": sub_list,
        "Quantity": np.random.randint(1, 10, n),
        "Unit Price": np.round(np.random.uniform(5, 500, n), 2),
        "Discount": np.random.choice([0, 0.05, 0.1, 0.2, 0.3], n),
    })
    raw["Sales"] = np.round(raw["Quantity"] * raw["Unit Price"] * (1 - raw["Discount"]), 2)

    # profit margin falls as discount rises (a realistic pattern to discover)
    margin = np.random.normal(0.30 - raw["Discount"] * 1.1, 0.06)
    raw["Profit"] = np.round(raw["Sales"] * margin, 2)

    # add messiness on purpose: missing regions and duplicate rows
    raw.loc[np.random.choice(n, 40, replace=False), "Region"] = np.nan
    raw = pd.concat([raw, raw.sample(25, random_state=1)], ignore_index=True)

    raw.to_csv(DATA_PATH, index=False)
    print("Dataset created:", DATA_PATH)
else:
    print("Using existing file:", DATA_PATH)

## Step 3: Load the data

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

## Step 4: Understand the data
Check the size, column types, and summary statistics. Note which columns are dates stored as text and which have missing values.

In [ ]:
print("Rows, Columns:", df.shape)
print()
df.info()

In [ ]:
df.describe()

## Step 5: Clean the data
### 5.1 Find problems

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

### 5.2 Fix them
- Remove duplicate rows.
- Fill missing `Region` with "Unknown" (we keep the rows because their sales are still valid).
- Convert `Order Date` from text to a real date.
- Tidy column names (spaces and hyphens to underscores).

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows")

df["Region"] = df["Region"].fillna("Unknown")

df["Order Date"] = pd.to_datetime(df["Order Date"])

df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("-", "_")

print("Missing values left:", df.isnull().sum().sum())
print("Columns:", df.columns.tolist())

### 5.3 Check for outliers
Large orders can be real, so we look at them but do not delete them automatically.

In [ ]:
sns.boxplot(x=df["Sales"])
plt.title("Sales Outlier Check")
plt.xlabel("Sales per order")
plt.show()

q1, q3 = df["Sales"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print(f"Orders above the outlier threshold ({upper:,.0f}): {(df['Sales'] > upper).sum()}")

## Step 6: Feature engineering
Create new columns that make the analysis easier.

In [ ]:
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Quarter"] = df["Order_Date"].dt.quarter
df["Day_of_Week"] = df["Order_Date"].dt.day_name()
df["Profit_Margin"] = df["Profit"] / df["Sales"]

df.head()

## Step 7: Exploratory analysis and charts
After each chart, look at it and write one sentence about what it shows. Those sentences become your insights.

### 7.1 Overall KPIs

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()

print(f"Total Sales:          {total_sales:,.2f}")
print(f"Total Profit:         {total_profit:,.2f}")
print(f"Overall Margin:       {total_profit / total_sales:.1%}")
print(f"Total Orders:         {df['Order_ID'].nunique():,}")
print(f"Average Order Value:  {df['Sales'].mean():,.2f}")

### 7.2 Monthly sales trend

In [ ]:
monthly = df.groupby(df["Order_Date"].dt.to_period("M"))["Sales"].sum()
monthly.index = monthly.index.to_timestamp()

plt.plot(monthly.index, monthly.values, marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("charts/01_monthly_sales_trend.png", dpi=150)
plt.show()

### 7.3 Yearly sales

In [ ]:
yearly = df.groupby("Year")["Sales"].sum().reset_index()

sns.barplot(data=yearly, x="Year", y="Sales")
plt.title("Total Sales by Year")
plt.tight_layout()
plt.savefig("charts/02_yearly_sales.png", dpi=150)
plt.show()

### 7.4 Seasonality: sales by month

In [ ]:
month_order = ["January", "February", "March", "April", "May", "June", "July",
               "August", "September", "October", "November", "December"]
season = df.groupby("Month_Name")["Sales"].sum().reindex(month_order).reset_index()

sns.barplot(data=season, x="Month_Name", y="Sales")
plt.xticks(rotation=45)
plt.title("Sales by Month (Seasonality)")
plt.tight_layout()
plt.savefig("charts/03_seasonality.png", dpi=150)
plt.show()

### 7.5 Sales and profit by category

In [ ]:
cat = df.groupby("Category")[["Sales", "Profit"]].sum().reset_index()
cat_long = cat.melt(id_vars="Category", var_name="Metric", value_name="Amount")

sns.barplot(data=cat_long, x="Category", y="Amount", hue="Metric")
plt.title("Sales vs Profit by Category")
plt.tight_layout()
plt.savefig("charts/04_category_sales_profit.png", dpi=150)
plt.show()

### 7.6 Sub-categories ranked by sales

In [ ]:
sub = (df.groupby("Sub_Category")["Sales"].sum()
         .sort_values(ascending=False).reset_index())

sns.barplot(data=sub, x="Sales", y="Sub_Category")
plt.title("Sub-Categories by Sales")
plt.tight_layout()
plt.savefig("charts/05_subcategories.png", dpi=150)
plt.show()

### 7.7 Regional performance

In [ ]:
region = df.groupby("Region")[["Sales", "Profit"]].sum().sort_values("Sales", ascending=False).reset_index()

sns.barplot(data=region, x="Region", y="Sales")
plt.title("Sales by Region")
plt.tight_layout()
plt.savefig("charts/06_region_sales.png", dpi=150)
plt.show()

region

### 7.8 Does discounting hurt profit?

In [ ]:
disc = df.groupby("Discount")["Profit"].mean().reset_index()

sns.barplot(data=disc, x="Discount", y="Profit")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Average Profit per Order by Discount Level")
plt.tight_layout()
plt.savefig("charts/07_discount_vs_profit.png", dpi=150)
plt.show()

disc

### 7.9 Correlation heatmap

In [ ]:
corr = df[["Sales", "Quantity", "Discount", "Profit"]].corr()

sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("charts/08_correlation_heatmap.png", dpi=150)
plt.show()

### 7.10 Distribution of order values

In [ ]:
sns.histplot(df["Sales"], bins=40, kde=True)
plt.title("Distribution of Sales per Order")
plt.xlabel("Sales per order")
plt.tight_layout()
plt.savefig("charts/09_sales_distribution.png", dpi=150)
plt.show()

### 7.11 Sales by day of week

In [ ]:
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = df.groupby("Day_of_Week")["Sales"].sum().reindex(dow_order).reset_index()

sns.barplot(data=dow, x="Day_of_Week", y="Sales")
plt.xticks(rotation=45)
plt.title("Sales by Day of Week")
plt.tight_layout()
plt.savefig("charts/10_day_of_week.png", dpi=150)
plt.show()

## Step 8: Key insights
This cell calculates the main findings from your data, so the numbers always match your charts. Copy them into your README and write the recommendation yourself.

In [ ]:
best_month = df.groupby("Month_Name")["Sales"].sum().idxmax()
best_cat_sales = df.groupby("Category")["Sales"].sum().idxmax()
best_cat_profit = df.groupby("Category")["Profit"].sum().idxmax()
best_region = df[df["Region"] != "Unknown"].groupby("Region")["Sales"].sum().idxmax()
worst_region = df[df["Region"] != "Unknown"].groupby("Region")["Sales"].sum().idxmin()
q_best = df.groupby("Quarter")["Sales"].sum().idxmax()

yearly_sales = df.groupby("Year")["Sales"].sum()
growth = yearly_sales.pct_change().dropna()

profit_by_disc = df.groupby("Discount")["Profit"].mean()
loss_levels = profit_by_disc[profit_by_disc <= 0].index.tolist()
loss_orders = (df["Profit"] < 0).mean()

print("KEY INSIGHTS")
print("-" * 60)
print(f"1. Peak month: {best_month}. Peak quarter: Q{q_best}.")
for y, g in growth.items():
    print(f"2. Sales growth in {y}: {g:+.1%} vs previous year.")
print(f"3. Highest sales category: {best_cat_sales}. Highest profit category: {best_cat_profit}.")
print(f"4. Top region: {best_region}. Lowest region: {worst_region}.")
if loss_levels:
    print(f"5. Average profit is zero or negative at discount level(s): {loss_levels}.")
else:
    print("5. Average profit stays positive at every discount level.")
print(f"6. {loss_orders:.1%} of all orders lost money.")
print(f"7. Correlation between discount and profit: {df['Discount'].corr(df['Profit']):.2f}")

### Recommendations (fill these in using your own results)
Write 3 to 5 bullets in this format: **Finding, Evidence (which chart), Recommendation.**

Example: *Discounts of 20% and above produce negative average profit (Chart 7). Recommend capping discounts at 10%.*

## Step 9: Save and download your charts
This zips the `charts` folder so you can download it from Colab and upload the images to GitHub.

In [ ]:
import shutil
shutil.make_archive("charts", "zip", "charts")
print("Created charts.zip with", len(os.listdir("charts")), "images")

try:
    from google.colab import files
    files.download("charts.zip")
    files.download("data/sales.csv")
except ImportError:
    print("Not running in Colab, so find charts.zip in the project folder.")